In [0]:
# Incremental data processing from volumes to bronze layer: Multiplex_bronze ingestion
from pyspark.sql import functions as F

def process_bronze():
    df = (
      spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .schema (schema="key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG")
          .option("pathGlobFilter", "*.json")
          .load("/Volumes/dev/landing_zone/kafka_source/book_store/")
          .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))
          .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
        .writeStream
          .option("checkpointLocation","/Volumes/dev/landing_zone/kafka_source/checkpoints/bronze/")
          .option("mergeSchema", True)
          .partitionBy("topic", "year_month")
          .trigger(availableNow=True)
          .table("dev.bookstore_bronze.bookstore_bronze")
    )
    df.awaitTermination()

process_bronze()
  